# Track 1 — Gobierno, Seguridad y Zero Trust
**Rol:** CRB_SEGURIDAD_INFO | **Tiempo:** 15 min | **Criterio:** Políticas automatizadas, clasificación PII, gobierno auditable

In [ ]:
USE ROLE CRB_SEGURIDAD_INFO;
USE DATABASE CREDIBANCO_HOL;
USE WAREHOUSE CREDIBANCO_HOL_WH;

## Bloque 1 — Evidencia: Gobierno activo sin intervención

In [ ]:
-- Datos PROTEGIDOS: el PAN y celular están enmascarados
SELECT comercio_id, PAN_SINTETICO, CELULAR, NOMBRE
FROM CREDIBANCO_HOL.CLIENTES.TARJETAHABIENTES LIMIT 5;

In [ ]:
-- Cambiar a rol de negocio: ¿qué ve?
USE ROLE CRB_NEGOCIO;
SELECT comercio_id, PAN_SINTETICO, CELULAR, NOMBRE
FROM CREDIBANCO_HOL.CLIENTES.TARJETAHABIENTES LIMIT 5;

In [ ]:
-- Volver al rol de seguridad
USE ROLE CRB_SEGURIDAD_INFO;
-- ¿Qué políticas protegen qué columnas?
SELECT * FROM TABLE(CREDIBANCO_HOL.INFORMATION_SCHEMA.POLICY_REFERENCES(
  REF_ENTITY_NAME => 'CREDIBANCO_HOL.CLIENTES.TARJETAHABIENTES',
  REF_ENTITY_DOMAIN => 'TABLE'
));

## Bloque 2 — Ejecutar: Clasificación automática de PII

In [ ]:
-- Snowflake detecta PII automáticamente (sin reglas manuales)
SELECT EXTRACT_SEMANTIC_CATEGORIES(
  'CREDIBANCO_HOL.CLIENTES.TARJETAHABIENTES'
);

In [ ]:
-- Auto-tagging: Snowflake clasifica Y aplica tags en un solo paso
CALL SYSTEM$CLASSIFY(
  'CREDIBANCO_HOL.CLIENTES.TARJETAHABIENTES',
  {'auto_tag': true}
);

In [ ]:
-- Verificar tags aplicados automáticamente
SELECT SYSTEM$GET_TAG('CREDIBANCO_HOL.GOBIERNO.TAG_PII',
  'CREDIBANCO_HOL.CLIENTES.TARJETAHABIENTES.CELULAR', 'COLUMN') AS tag_celular;

## Bloque 3 — CoCo
Copia este prompt en Cortex Code:

> **Aplica Zero Trust a la tabla TARJETAHABIENTES: clasifica PII automáticamente, crea masking para PAN y celular, aplica row access policy, y verifica que CRB_NEGOCIO no pueda ver datos sensibles. Muéstrame la evidencia.**

In [ ]:
-- Verificación final
SELECT 'T1_COMPLETO' AS status,
  (SELECT COUNT(*) FROM TABLE(CREDIBANCO_HOL.INFORMATION_SCHEMA.POLICY_REFERENCES(
    REF_ENTITY_NAME => 'CREDIBANCO_HOL.CLIENTES.TARJETAHABIENTES', REF_ENTITY_DOMAIN => 'TABLE'))) AS politicas_activas;